In [ ]:
"""
CASA0006 Coursework - Data Preprocessing Script 
Filters STATS19 road safety data for Greater London (2022-2023)
Handles new collision_index naming convention
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ========== Configuration ==========
RAW_DATA_DIR = Path("data/raw")
PROCESSED_DATA_DIR = Path("data/processed")
BOUNDARIES_DIR = Path("data/boundaries")

# Ensure output directories exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
BOUNDARIES_DIR.mkdir(parents=True, exist_ok=True)

# Years to process
YEARS = [2022, 2023]

# London police force codes
LONDON_POLICE_FORCES = [1, 48]  # 1=Metropolitan Police, 48=City of London Police

print("=" * 60)
print("CASA0006 - London Road Safety Data Preprocessing")
print("=" * 60)

# ========== Step 1: Load and filter collision data ==========
print("\n[1/6] Loading collision data...")

collision_dfs = []

for year in YEARS:
    filepath = RAW_DATA_DIR / f"dft-road-casualty-statistics-collision-{year}.csv"
    print(f"  Reading: {filepath.name}")
    
    df = pd.read_csv(filepath, low_memory=False)
    
    # Filter for London
    df_london = df[df['police_force'].isin(LONDON_POLICE_FORCES)].copy()
    
    collision_dfs.append(df_london)
    print(f"    → {len(df_london):,} London collisions in {year}")

# Combine years
collisions_london = pd.concat(collision_dfs, ignore_index=True)
print(f"\n  Total London collisions (2022-2023): {len(collisions_london):,}")

# Get unique collision indices for filtering other tables
london_collision_indices = set(collisions_london['collision_index'])

# ========== Step 2: Load and filter casualty data ==========
print("\n[2/6] Loading casualty data...")

casualty_dfs = []
for year in YEARS:
    filepath = RAW_DATA_DIR / f"dft-road-casualty-statistics-casualty-{year}.csv"
    print(f"  Reading: {filepath.name}")
    
    df = pd.read_csv(filepath, low_memory=False)
    
    # Filter by matching collision_index with London collisions
    df_london = df[df['collision_index'].isin(london_collision_indices)].copy()
    
    casualty_dfs.append(df_london)
    print(f"    → {len(df_london):,} casualties in London {year}")

casualties_london = pd.concat(casualty_dfs, ignore_index=True)
print(f"\n  Total London casualties (2022-2023): {len(casualties_london):,}")

# ========== Step 3: Load and filter vehicle data ==========
print("\n[3/6] Loading vehicle data...")

vehicle_dfs = []
for year in YEARS:
    filepath = RAW_DATA_DIR / f"dft-road-casualty-statistics-vehicle-{year}.csv"
    print(f"  Reading: {filepath.name}")
    
    df = pd.read_csv(filepath, low_memory=False)
    
    # Filter by matching collision_index
    df_london = df[df['collision_index'].isin(london_collision_indices)].copy()
    
    vehicle_dfs.append(df_london)
    print(f"    → {len(df_london):,} vehicles in London {year}")

vehicles_london = pd.concat(vehicle_dfs, ignore_index=True)
print(f"\n  Total London vehicles (2022-2023): {len(vehicles_london):,}")

# ========== Step 4: Data quality checks ==========
print("\n[4/6] Running data quality checks...")

# Check for missing collision_index (should be zero)
missing_indices = [
    ("collisions", collisions_london['collision_index'].isna().sum()),
    ("casualties", casualties_london['collision_index'].isna().sum()),
    ("vehicles", vehicles_london['collision_index'].isna().sum())
]

for table, count in missing_indices:
    if count > 0:
        print(f"  ⚠️  Warning: {count} rows with missing collision_index in {table}")
    else:
        print(f"  ✓ {table}: no missing collision_index")

# Check severity distribution
severity_counts = collisions_london['collision_severity'].value_counts().sort_index()
print(f"\n  Collision severity distribution:")
severity_mapping = {1: "Fatal", 2: "Serious", 3: "Slight"}
for severity, count in severity_counts.items():
    label = severity_mapping.get(severity, f"Unknown ({severity})")
    pct = 100 * count / len(collisions_london)
    print(f"    {label}: {count:,} ({pct:.1f}%)")

# ========== Step 5: Create binary target variable ==========
print("\n[5/6] Creating binary target variable...")

# Binary classification: Serious+Fatal (1,2) vs Slight (3)
collisions_london['severity_binary'] = (
    collisions_london['collision_severity'].isin([1, 2])
).astype(int)

binary_counts = collisions_london['severity_binary'].value_counts()
print(f"  Class distribution:")
print(f"    Slight (0): {binary_counts.get(0, 0):,} ({100*binary_counts.get(0, 0)/len(collisions_london):.1f}%)")
print(f"    Serious/Fatal (1): {binary_counts.get(1, 0):,} ({100*binary_counts.get(1, 0)/len(collisions_london):.1f}%)")

if binary_counts.get(1, 0) > 0:
    imbalance_ratio = binary_counts.get(0, 0) / binary_counts.get(1, 0)
    print(f"  → Imbalance ratio: {imbalance_ratio:.2f}:1")

# ========== Step 6: Save processed data ==========
print("\n[6/6] Saving processed datasets...")

output_files = {
    "london_collisions_2022_2023.csv": collisions_london,
    "london_casualties_2022_2023.csv": casualties_london,
    "london_vehicles_2022_2023.csv": vehicles_london
}

for filename, df in output_files.items():
    output_path = PROCESSED_DATA_DIR / filename
    df.to_csv(output_path, index=False)
    
    # Check file size
    size_mb = output_path.stat().st_size / (1024 * 1024)
    
    if size_mb > 100:
        print(f"  ⚠️  {filename}: {size_mb:.1f} MB (EXCEEDS GitHub 100MB limit!)")
        print(f"      → Need to reduce size or use external hosting")
    elif size_mb > 50:
        print(f"  ⚠️  {filename}: {size_mb:.1f} MB (approaching limit)")
    else:
        print(f"  ✓ {filename}: {len(df):,} rows, {size_mb:.1f} MB")

# ========== Bonus: Create Inner/Outer London classification ==========
print("\n[Bonus] Creating Inner/Outer London borough classification...")

# ONS official classification (aligned with local_authority_district field)
inner_london_boroughs = [
    "Camden", "Greenwich", "Hackney", "Hammersmith and Fulham", "Islington",
    "Kensington and Chelsea", "Lambeth", "Lewisham", "Southwark", "Tower Hamlets",
    "Wandsworth", "Westminster", "City of London"
]

outer_london_boroughs = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley", "Croydon",
    "Ealing", "Enfield", "Haringey", "Harrow", "Havering", "Hillingdon", "Hounslow",
    "Kingston upon Thames", "Merton", "Newham", "Redbridge", "Richmond upon Thames",
    "Sutton", "Waltham Forest"
]

borough_classification = pd.DataFrame({
    'local_authority_district': inner_london_boroughs + outer_london_boroughs,
    'london_region': ['Inner London'] * len(inner_london_boroughs) + 
                     ['Outer London'] * len(outer_london_boroughs)
})

boundary_file = BOUNDARIES_DIR / "london_boroughs_inner_outer.csv"
borough_classification.to_csv(boundary_file, index=False)
print(f"  ✓ Saved: {boundary_file.name} ({len(borough_classification)} boroughs)")

# Print actual borough names in data to verify matching
print(f"\n  Borough names in collision data (first 10):")
actual_boroughs = collisions_london['local_authority_district'].value_counts().head(10)
for borough, count in actual_boroughs.items():
    print(f"    {borough}: {count:,} collisions")

# ========== Final Summary ==========
print("\n" + "=" * 60)
print("✓ Preprocessing complete!")
print("=" * 60)
print(f"\nProcessed files saved to: {PROCESSED_DATA_DIR.absolute()}")
print(f"Boundary data saved to: {BOUNDARIES_DIR.absolute()}")

print(f"\nKey collision columns available for modeling:")
modeling_cols = [
    'collision_severity', 'severity_binary', 'speed_limit', 'road_type', 
    'junction_detail', 'light_conditions', 'weather_conditions', 
    'road_surface_conditions', 'urban_or_rural_area', 'number_of_vehicles',
    'local_authority_district', 'collision_year'
]
available = [col for col in modeling_cols if col in collisions_london.columns]
print(f"  Available ({len(available)}/{len(modeling_cols)}): {', '.join(available[:8])}...")

print("\nNext steps:")
print("1. Verify all files are under 100MB")
print("2. Upload to GitHub: data/processed/*.csv and data/boundaries/*.csv")
print("3. Make repo Public")
print("4. Get raw URLs for notebook remote reading")
print("5. Share your GitHub username for notebook template completion")

CASA0006 - London Road Safety Data Preprocessing

[1/6] Loading collision data...
  Reading: dft-road-casualty-statistics-collision-2022.csv
    → 23,502 London collisions in 2022
  Reading: dft-road-casualty-statistics-collision-2023.csv
    → 22,914 London collisions in 2023

  Total London collisions (2022-2023): 46,416

[2/6] Loading casualty data...
  Reading: dft-road-casualty-statistics-casualty-2022.csv
    → 27,259 casualties in London 2022
  Reading: dft-road-casualty-statistics-casualty-2023.csv
    → 26,210 casualties in London 2023

  Total London casualties (2022-2023): 53,469

[3/6] Loading vehicle data...
  Reading: dft-road-casualty-statistics-vehicle-2022.csv
    → 42,783 vehicles in London 2022
  Reading: dft-road-casualty-statistics-vehicle-2023.csv
    → 41,536 vehicles in London 2023

  Total London vehicles (2022-2023): 84,319

[4/6] Running data quality checks...
  ✓ collisions: no missing collision_index
  ✓ casualties: no missing collision_index
  ✓ vehicles: 